<a href="https://colab.research.google.com/github/Alyc3/Alyc3/blob/main/Modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/youngseo0526/X-AVDT.git
%cd X-AVDT

Cloning into 'X-AVDT'...
remote: Enumerating objects: 279, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 279 (delta 8), reused 3 (delta 3), pack-reused 263 (from 1)
Receiving objects: 100% (279/279), 32.92 MiB | 33.92 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/X-AVDT


In [18]:
# Instalar virtualenv que sí funciona en Colab
!pip install virtualenv -q

# Crear entorno con virtualenv
!virtualenv /content/xavdt_env -p python3.10 -q

VENV = "/content/xavdt_env"

# Verificar que pip existe
!ls {VENV}/bin/pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 39.1 MB/s eta 0:00:00
/content/xavdt_env/bin/pip


In [19]:
VENV = "/content/xavdt_env"

# Versiones fijas primero
!{VENV}/bin/pip install tokenizers==0.13.3 transformers==4.26.1 diffusers==0.27.2 accelerate==0.28.0 huggingface-hub==0.25.2 -q

# Torch con CUDA
!{VENV}/bin/pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

# Resto de dependencias
!{VENV}/bin/pip install numpy scikit-learn tqdm Pillow matplotlib omegaconf safetensors einops opencv-python mediapipe insightface onnxruntime-gpu audio-separator librosa soundfile moviepy pytorch-metric-learning av -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
# Paso 1: sistema
!apt-get install -y ffmpeg git-lfs -q
!git lfs install

# Paso 2: instalar Rust (necesario para compilar tokenizers antiguo)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] += ":/root/.cargo/bin"

# Paso 3: fijar versiones problemáticas PRIMERO
!pip install tokenizers==0.13.3 -q
!pip install transformers==4.26.1 --no-deps -q

# Paso 4: instalar torch con CUDA
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

# Paso 5: instalar el resto de requirements evitando sobreescribir lo ya instalado
!pip install -r requirements.txt --no-deps -q

# Paso 6: instalar dependencias que no conflictúan
!pip install numpy scikit-learn tqdm Pillow matplotlib omegaconf \
             diffusers==0.27.2 accelerate==0.28.0 huggingface-hub==0.25.2 \
             safetensors einops opencv-python mediapipe \
             insightface onnxruntime-gpu audio-separator \
             librosa soundfile moviepy pytorch-metric-learning av -q

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
Updated git hooks.
Git LFS initialized.
info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-07-09 for version 1.97.0 (2d8144b78 2026-07-07)
info: downloading 6 components
        cargo downloading [               ]         0 B (0 B/s, ETA: 0s)
        cargo downloading [               ]   10.63 MiB (0 B/s, ETA: 0s)
        cargo 

In [5]:
%cd hallo
!git clone https://huggingface.co/fudan-generative-ai/hallo pretrained_models
%cd ..

/content/X-AVDT/hallo
Cloning into 'pretrained_models'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 61 (delta 6), reused 1 (delta 1), pack-reused 54 (from 1)
Receiving objects: 100% (61/61), 17.21 KiB | 17.21 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Filtering content: 100% (12/12), 6.53 GiB | 24.90 MiB/s, done.
Encountered 1 file(s) that may not have been copied correctly on Windows:
	hallo/net.pth

See: `git lfs help smudge` for more details.
/content/X-AVDT


In [6]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
import os

os.makedirs("results/x_avdt", exist_ok=True)
os.makedirs("my_videos", exist_ok=True)

# Checkpoint - ajusta el nombre exacto de tu archivo .pt
!cp "/content/drive/MyDrive/Modelo/sa_triplet_dec_bs8_800_es.pt" results/x_avdt/model_best.pt

# Video con espacio en el nombre - comillas dobles alrededor de la ruta completa
!cp "/content/drive/MyDrive/Video Alterado.mp4" my_videos/

In [9]:
# Confirmar que el checkpoint existe
!ls -lh results/x_avdt/

# Confirmar que el video está ahí
!ls -lh my_videos/

total 181M
-rw------- 1 root root 181M Jul 16 03:28 model_best.pt
total 1.3M
-rw------- 1 root root 1.3M Jul 16 03:28 'Video Alterado.mp4'


In [10]:
!python hallo/preprocess_videos.py extract-frames \
  --video_dir my_videos \
  --frames_dir my_frames \
  --duration 5 \
  --fps 25 \
  --size 512 512

Extracting raw videos: 100% 1/1 [00:03<00:00,  3.89s/it]


In [13]:
# Forzar versiones exactas requeridas por X-AVDT
!pip install --force-reinstall \
    tokenizers==0.13.3 \
    transformers==4.26.1 \
    diffusers==0.27.2 \
    accelerate==0.28.0 \
    huggingface-hub==0.25.2 -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 547.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.3/801.3 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6

In [14]:
import tokenizers, transformers, diffusers, accelerate
print("tokenizers:", tokenizers.__version__)      # debe ser 0.13.3
print("transformers:", transformers.__version__)   # debe ser 4.26.1
print("diffusers:", diffusers.__version__)         # debe ser 0.27.2
print("accelerate:", accelerate.__version__)       # debe ser 0.28.0

ImportError: tokenizers>=0.11.1,!=0.11.3,<0.14 is required for a normal functioning of this module, but found tokenizers==0.22.2.
Try: pip install transformers -U or pip install -e '.[dev]' if you're working with git main

In [11]:
!python hallo/extract_features.py \
  --frames_dir my_frames \
  --output_dir my_hallo_features

2026-07-16 03:30:27.159651: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
Moving 0 files to the new cache system
0it [00:00, ?it/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/diffusers/utils/import_utils.py", line 718, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_impor

In [12]:
!python train/evaluate.py \
  --data_dir my_pt_features/ \
  --ckpt results/x_avdt/model_best.pt

Checkpoint: results/x_avdt/model_best.pt
Traceback (most recent call last):
  File "/content/X-AVDT/train/evaluate.py", line 302, in <module>
    main()
  File "/content/X-AVDT/train/evaluate.py", line 145, in main
    dataset = InversionDataset(data_dir=args.data_dir, split="test")
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/X-AVDT/train/dataset.py", line 59, in __init__
    raise FileNotFoundError(f"Real directory not found: {real_dir}")
FileNotFoundError: Real directory not found: my_pt_features/test/real
